# Generador de clientes sintéticos pensionarios de México

Crea una base de personas ficticias (IMSS Ley 73 / Ley 97, ISSSTE Cuentas Individuales e independientes sin afiliación) a partir de los CSV del directorio de entrada. No utiliza ni produce datos personales reales.

- **Parámetros:** todo lo ajustable (rutas, semilla, `N_CLIN`, fecha de corte, proporción por perfil, distribuciones de edad, salario, saldos, probabilidades y montos legales de referencia) vive en `config/parametros.py`.
- **Lógica:** paquete `generador/`. Cada cliente se genera por perfil, se clasifica con reglas pensionarias y se valida al 100 %.
- `N_CLIN` es exactamente el número de clientes a generar.
- Los importes están en pesos constantes de la fecha de corte. Todas las resoluciones son simuladas.

In [12]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Funciona si Jupyter se abre desde la raíz del proyecto o desde esta carpeta.
RUTA_NOTEBOOK = next(
    (ruta for ruta in (Path.cwd(), Path.cwd() / 'SyntethicDataGenerator') if (ruta / 'config').is_dir()),
    None,
)
if RUTA_NOTEBOOK is None:
    raise FileNotFoundError('No se encontró el directorio config junto al notebook.')
sys.path.insert(0, str(RUTA_NOTEBOOK))

# Descarta módulos en caché para que los cambios en config/ se apliquen al volver a ejecutar.
for modulo in [m for m in sys.modules if m.split('.')[0] in ('config', 'generador')]:
    del sys.modules[modulo]

import config as cfg
from generador import generar_clientes

cfg.DIR_OUTPUT.mkdir(exist_ok=True)
rng = np.random.default_rng(cfg.SEMILLA)

In [13]:
nombres = pd.read_csv(cfg.ARCHIVO_NOMBRES, encoding=cfg.CODIFICACION_ENTRADA)
apellidos = pd.read_csv(cfg.ARCHIVO_APELLIDOS, encoding=cfg.CODIFICACION_ENTRADA)

# Controles de calidad de los insumos.
assert len(nombres) == cfg.NOMBRES_ESPERADOS, f'Se esperan {cfg.NOMBRES_ESPERADOS} nombres.'
assert set(nombres['genero']) == set(cfg.GENEROS_ARCHIVO), 'Géneros inesperados en el archivo de nombres.'
assert len(apellidos) == cfg.APELLIDOS_ESPERADOS, f'Se esperan {cfg.APELLIDOS_ESPERADOS} apellidos.'

display(nombres.groupby('genero').size().rename('cantidad'))
display(apellidos.head())

genero
Hombre    250
Mujer     250
Name: cantidad, dtype: int64

,id,apellido
0,1,Hernández
1,2,García
2,3,Martínez
3,4,López
4,5,González


In [14]:
# Genera historial laboral, saldos (salario, RCV, AV, vivienda), pensión estimada e identificadores.
clientes = generar_clientes(rng, nombres, apellidos)
clientes.head(10)

,id_cliente,nombre,ap_paterno,ap_materno,age,f_nacim,genero,entidad,nss,num_issste,...,saldo_rcv,saldo_ahorro_solidario,porcentaje_ahorro_solidario,saldo_bono_pension,saldo_av,saldo_vivienda,av_mensual,regimen_estimado,ley_bandera,escenario_estimado
0,1,Christopher,Parra,Linares,64,1962-06-30,H,BC,69886236681,None,...,848971.77,0.00,NaN,0.0,0.00,0.00,0.00,IMSS73,LEY_73,IMSS73_DERECHO_PENSION
1,2,Samuel,Cabrera,Chavarría,68,1958-06-18,H,CC,None,35955841636,...,217296.49,3816.99,1.756580,0.0,0.00,211546.39,478.96,ISSSTE,LEY_ISSSTE,ISSSTE_NEGATIVA_PENSION
2,3,Anabel,Santana,Labastida,64,1961-12-28,M,QT,None,54846101225,...,149990.87,2574.87,1.716685,350000.0,91455.69,0.00,0.00,ISSSTE,LEY_ISSSTE,ISSSTE_PENSION_GARANTIZADA
3,4,Irene,Ortiz,Ortega,62,1964-02-12,M,TL,12836468285,None,...,152100.51,0.00,NaN,0.0,39662.32,200769.39,0.00,IMSS73,LEY_73,IMSS73_GARANTIZADA
4,5,Miguel,Valencia,Peña,62,1963-12-12,H,CH,32006357043,None,...,1072632.64,0.00,NaN,0.0,0.00,714012.84,0.00,IMSS97,LEY_97,IMSS97_RECURSOS_PROPIOS
5,6,Amado,León,García,60,1966-02-05,H,DF,85866645444,None,...,205305.02,0.00,NaN,0.0,0.00,141599.60,0.00,IMSS73,LEY_73,IMSS73_GARANTIZADA
6,7,Alán,Leal,Galindo,65,1961-02-26,H,CS,68046104144,None,...,281787.21,0.00,NaN,0.0,28592.91,1577.14,0.00,IMSS97,LEY_97,IMSS97_PENSION_GARANTIZADA
7,8,Gabriela,Miranda,Gamez,60,1966-05-11,M,TL,47086620029,None,...,147583.63,0.00,NaN,0.0,0.00,126419.45,0.00,IMSS97,LEY_97,IMSS97_NEGATIVA_PENSION
8,9,Jan,Monroy,Rodríguez,66,1960-09-09,H,CS,None,45906015461,...,925356.80,18337.27,1.981643,350000.0,0.00,0.00,283.45,ISSSTE,LEY_ISSSTE,ISSSTE_RECURSOS_PROPIOS
9,10,Yahir,Navarro,Rivera,69,1957-01-22,H,VZ,28975728859,None,...,631824.14,0.00,NaN,0.0,20953.48,0.00,0.00,IMSS97,LEY_97,IMSS97_PENSION_GARANTIZADA


In [15]:
# Control de calidad del layout final y de los identificadores sintéticos.
assert list(clientes.columns) == cfg.COLUMNAS_SALIDA
assert clientes['id_cliente'].is_unique
assert clientes['curp'].is_unique
assert clientes['correo'].is_unique
assert len(clientes) == cfg.N_CLIN
assert clientes['correo'].str.endswith(('@yahoo.com', '@gmail.com', '@hotmail.com', '@outlook.com')).all()
assert clientes.loc[clientes['nss'].notna(), 'nss'].str.fullmatch(r'\d{11}').all()
assert clientes['anios_cotizados'].ge(0).all()

print('Régimen y escenario estimado')
display(clientes.groupby(['regimen_estimado', 'escenario_estimado']).size().rename('clientes'))
display(pd.crosstab(pd.cut(clientes['age'], [0, 49, 59, 64, 69, 120], labels=['<50', '50-59', '60-64', '65-69', '70+']),
                    clientes['regimen_estimado']))
display(clientes[['age', 'anios_cotizados', 'sem_cot', 'salario_mensual', 'saldo_rcv', 'saldo_av', 'saldo_vivienda', 'av_mensual']].describe(percentiles=[.5, .9, .99]).round(1))
print(f"Clientes con AV = 0: {(clientes['saldo_av'] == 0).mean():.1%} | con vivienda = 0: {(clientes['saldo_vivienda'] == 0).mean():.1%}")

Régimen y escenario estimado


regimen_estimado  escenario_estimado        
IMSS73            IMSS73_DERECHO_PENSION        15000
                  IMSS73_GARANTIZADA            10000
                  IMSS73_NEGATIVA_PENSION        6000
IMSS97            IMSS97_NEGATIVA_PENSION        8000
                  IMSS97_PENSION_GARANTIZADA    17000
                  IMSS97_RECURSOS_PROPIOS        8000
ISSSTE            ISSSTE_NEGATIVA_PENSION        6000
                  ISSSTE_PENSION_GARANTIZADA    10000
                  ISSSTE_RECURSOS_PROPIOS       10000
NO_APLICA         INDEPENDIENTES                10000
Name: clientes, dtype: int64

regimen_estimado,IMSS73,IMSS97,ISSSTE,NO_APLICA
age,,,,
<50,0,0,0,4262
50-59,0,0,0,3913
60-64,23756,24957,15734,1217
65-69,7034,7951,9433,608
70+,210,92,833,0


,age,anios_cotizados,sem_cot,salario_mensual,saldo_rcv,saldo_av,saldo_vivienda,av_mensual
count,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0
mean,62.0,19.8,1055.8,21328.1,445171.9,28311.6,135768.4,25.9
std,5.1,10.2,534.6,12198.9,408308.8,151568.2,176045.7,140.3
min,30.0,0.0,0.0,9582.1,0.0,0.0,0.0,0.0
50%,63.0,22.0,1163.0,17737.4,328385.6,0.0,70458.0,0.0
90%,66.0,31.0,1620.1,37777.5,1053887.9,31310.3,395374.5,0.0
99%,70.0,36.0,1912.0,63474.9,1743059.9,750302.5,723458.6,776.8
max,74.0,45.0,2398.0,83606.0,4122611.8,6841068.2,1242999.6,1991.3


Clientes con AV = 0: 86.5% | con vivienda = 0: 35.7%


In [16]:
marca_tiempo = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S_%f')
carpeta_salida = cfg.DIR_OUTPUT / marca_tiempo
carpeta_salida.mkdir(parents=True, exist_ok=False)

archivos_por_regimen = {
    'IMSS97': 'clientes_IMSS97.csv',
    'IMSS73': 'clientes_IMSS73.csv',
    'ISSSTE': 'clientes_ISSSTE_Cuentas_Individuales.csv',
}
for regimen, nombre_archivo in archivos_por_regimen.items():
    grupo = clientes.loc[clientes['regimen_estimado'].eq(regimen)]
    ruta_archivo = carpeta_salida / nombre_archivo
    grupo.to_csv(ruta_archivo, index=False, encoding=cfg.CODIFICACION_SALIDA, date_format=cfg.FORMATO_FECHA)
    print(f'{len(grupo):,} registros: {ruta_archivo.resolve()}')

independientes = clientes['regimen_estimado'].eq(cfg.NO_APLICA).sum()
print(f'{independientes:,} independientes sin régimen contributivo no incluidos en los tres archivos.')

33,000 registros: /Users/alejandro_aac/Documents/WebScrapping/SyntethicDataGenerator/DataOutput/20261021_172249_771771/clientes_IMSS97.csv
31,000 registros: /Users/alejandro_aac/Documents/WebScrapping/SyntethicDataGenerator/DataOutput/20261021_172249_771771/clientes_IMSS73.csv
26,000 registros: /Users/alejandro_aac/Documents/WebScrapping/SyntethicDataGenerator/DataOutput/20261021_172249_771771/clientes_ISSSTE_Cuentas_Individuales.csv
10,000 independientes sin régimen contributivo no incluidos en los tres archivos.
